In [1]:
%run notebook_setup.ipynb

2025-07-05 04:24:07 - autoreload enabled
2025-07-05 04:24:14 - repo_dir set to /Users/rj/personal/GenePT-tools
2025-07-05 04:24:23 - data_dir set to /Users/rj/personal/GenePT-tools/data


File already exists at /Users/rj/personal/GenePT-tools/data/GenePT_emebdding_v2.zip
Extracting files...
Extracting GenePT_emebdding_v2/
Skipping GenePT_emebdding_v2/NCBI_UniProt_summary_of_genes.json - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_embedding_ada_text.pickle - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_protein_embedding_model_3_text.pickle. - already exists with same size
Skipping GenePT_emebdding_v2/NCBI_summary_of_genes.json - already exists with same size
Extraction complete!
Skipping embedding_original_ada_text.parquet - already exists
Skipping embedding_original_large_3.parquet - already exists
Skipping embedding_associations_age_cell_type_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_age_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_tissue_drug_pat

In [2]:
import pandas as pd

metadata_pdf = pd.read_parquet(data_dir / "cellxgene_v2_metadata_v2.parquet")

In [3]:
import json

def aggregate_obs_counts(pdf, obs_count_name):
  all_obs_counts = {}
  for json_obj in pdf.obs_counts:
    obs_counts = json.loads(json_obj[obs_count_name])
    for key, value in obs_counts.items():
      if key in all_obs_counts:
        all_obs_counts[key] += value
      else:
        all_obs_counts[key] = value

  obs_counts_pdf = pd.DataFrame(all_obs_counts.items(), columns=[obs_count_name, 'count']).set_index(obs_count_name).sort_values('count', ascending=False)
  return obs_counts_pdf

cell_type_counts_pdf = aggregate_obs_counts(metadata_pdf, 'cell_type')
cell_type_counts_pdf.head(50)

,count
cell_type,
neuron,7271693
L2/3-6 intratelencephalic projecting glutamatergic neuron,4310965
oligodendrocyte,3556599
fibroblast,2267012
unknown,2159597
"CD4-positive, alpha-beta T cell",2045255
"CD8-positive, alpha-beta T cell",1952771
macrophage,1950346
classical monocyte,1591535


# Filter criteria
* cut at 500
* downsample to 10K cells per cell type
* downsample across multiple data sets
* ~~preferentially select from datasets with summary statistics~~
* remove anything not 10x - `adata.obs['assay'].str.startswith('10x')`
* ~~filter large files if possible~~
* remove non-human - `adata.obs['organism'] == "Homo sapiens"`
* check for duplicates
* check for duplicates between training and test set

1. Filter
2. 

In [4]:
import pandas as pd

human_only_x10_cell_counts_pdf = pd.read_parquet(data_dir / "cellxgene" / "human_only_x10_cell_counts.parquet")
human_only_x10_metadata_pdf = pd.read_parquet(data_dir / "cellxgene" / "human_only_x10_metadata.parquet")
test_file_indices_pdf = pd.read_parquet(data_dir / "cellxgene" / "test_file_indices.parquet")
test_file_indices = test_file_indices_pdf.index

descriptions_pdf = pd.read_parquet(data_dir / "cellxgene" / "descriptions.parquet")

In [5]:
import pulp
import numpy as np

cell_types = human_only_x10_cell_counts_pdf.columns[:20]
file_indices = test_file_indices_pdf.index.tolist()
target_per_type = 1000

# Define the problem
prob = pulp.LpProblem("CellTypeCoverage", pulp.LpMinimize)

# Decision variables: x_i = 1 if file i is selected
x = pulp.LpVariable.dicts('select_file', file_indices, cat='Binary')

# Objective: Minimize total number of files

prob += pulp.lpSum([human_only_x10_metadata_pdf.iloc[i]['cell_count'] * x[i] for i in file_indices])

# Constraints: For each cell type, cover at least target_per_type cells
for ct in cell_types:
  prob += pulp.lpSum([human_only_x10_cell_counts_pdf.loc[i, ct] * x[i] for i in file_indices]) >= target_per_type

# Solve
prob.solve()

# Get selected files
selected_files = [i for i in file_indices if x[i].varValue > 0.5]
print("Selected files:", selected_files)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/1w/njpw08_93h73169nbj9b9z700000gp/T/68e6b12c0ae541c383db75cd97d12ab7-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/1w/njpw08_93h73169nbj9b9z700000gp/T/68e6b12c0ae541c383db75cd97d12ab7-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 25 COLUMNS
At line 719 RHS
At line 740 BOUNDS
At line 841 ENDATA
Problem MODEL has 20 rows, 100 columns and 393 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 34250 - 0.00 seconds
Cgl0004I processed model has 20 rows, 95 columns (95 integer (95 of which binary)) and 393 elements
Cbc0038I Initial state - 6 integers unsatisfied sum - 1.30613
Cbc0038I Pass   1: suminf.    0.32300 (2) obj. 359025 iterations 17
Cbc003

In [7]:
human_only_x10_cell_counts_pdf[cell_types]

,neuron,L2/3-6 intratelencephalic projecting glutamatergic neuron,oligodendrocyte,fibroblast,unknown,"CD4-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",macrophage,classical monocyte,T cell,radial glial cell,natural killer cell,B cell,glutamatergic neuron,malignant cell,endothelial cell,"naive thymus-derived CD4-positive, alpha-beta T cell",astrocyte,monocyte,alveolar macrophage
0,9022,0,635,25,0,0,0,0,0,0,0,0,0,0,0,16,0,141,0,0
1,0,5399,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,158,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,19391,6532,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,80211,7716,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,155448,107583,0,159044,0,0,62210,62548,0,0,0,0,0,0,0
9,0,0,0,0,177,0,0,0,0,0,0,0,124,0,0,0,0,0,0,0


In [6]:
print(metadata_pdf.index)
print(descriptions_pdf.index)

cominbed_metadata_pdf = pd.concat([metadata_pdf, descriptions_pdf], axis=1)

RangeIndex(start=0, stop=961, step=1)
RangeIndex(start=0, stop=961, step=1)


# selection criteria

* 10K cells for each cell type with enough
  * select uniforming across files 
* hold out 10% for testing for other types
* filter out tabula sapiens 
* test set should be past 2023-05-08 when scGPT was trained
  * 393 files
  * 152 genes in our list are not represented in these files 


In [7]:


def construct_counts_pdf(selected_files, cell_counts_pdf, cell_max):
  
  total_counts = []

  for cell_type in cell_counts_pdf.columns:
    if cell_type == 'unknown':
        continue
    # Exclude test indices for training
    train_df = cell_counts_pdf.iloc[selected_files]
    # Only keep rows where this cell type has > 0 cells
    counts_pdf = train_df[train_df[cell_type] > 0][[cell_type]].reset_index(drop=False)
    counts_pdf.columns = ['index', 'cell_count']
    counts_pdf['cell_type'] = cell_type

    total_cell_count = counts_pdf['cell_count'].sum()
    num_rows = len(counts_pdf)

    if total_cell_count <= cell_max:
        # If under or equal to 2,000, take all
        selected = counts_pdf.copy()
    else:
        # Evenly divide 2,000 among rows
        base_share = cell_max // num_rows
        remainder = cell_max % num_rows

        # Assign base_share to each row, and distribute the remainder
        counts_pdf['take_cells'] = base_share
        if remainder > 0:
            counts_pdf.loc[counts_pdf.index[:remainder], 'take_cells'] += 1

        # For each row, take up to the minimum of take_cells and available cells
        counts_pdf['cell_count'] = counts_pdf[['take_cells', 'cell_count']].min(axis=1)

        # If some rows can't provide their full share, redistribute the leftover
        leftover = cell_max - counts_pdf['cell_count'].sum()
        while leftover > 0:
            # Find rows that still have available cells
            mask = counts_pdf['cell_count'] < train_df.loc[counts_pdf['index'], cell_type].values
            if not mask.any():
                break  # No more cells to take
            for idx in counts_pdf[mask].index:
                available = train_df.loc[counts_pdf.at[idx, 'index'], cell_type]
                if counts_pdf.at[idx, 'cell_count'] < available:
                    counts_pdf.at[idx, 'cell_count'] += 1
                    leftover -= 1
                    if leftover == 0:
                        break

        selected = counts_pdf[['index', 'cell_type', 'cell_count']]

    total_counts.append(selected)

  return pd.concat(total_counts)

In [8]:
test_counts_pdf = construct_counts_pdf(selected_files, human_only_x10_cell_counts_pdf, 2000)
test_counts_pdf

,index,cell_type,cell_count
0,542,neuron,105
1,619,neuron,1895
0,704,L2/3-6 intratelencephalic projecting glutamate...,2000
0,704,oligodendrocyte,632
1,542,oligodendrocyte,632
2,619,oligodendrocyte,632
3,117,oligodendrocyte,104
0,27,fibroblast,1926
1,24,fibroblast,74
0,153,"CD4-positive, alpha-beta T cell",400


In [9]:
import pulp

def select_row_groups_ilp(cell_type_counts_per_row_group, needed):
  """
  cell_type_counts_per_row_group: list of dicts, one per row group, mapping cell_type to count
  needed: dict mapping cell_type to required count
  Returns: set of row group indices to read
  """
  n_row_groups = len(cell_type_counts_per_row_group)
  cell_types = list(needed.keys())

  # Define the problem
  prob = pulp.LpProblem("RowGroupSelection", pulp.LpMinimize)

  # Binary variables: x_r = 1 if row group r is selected
  x = [pulp.LpVariable(f"x_{r}", cat="Binary") for r in range(n_row_groups)]

  # Objective: minimize number of row groups
  prob += pulp.lpSum(x)

  # Constraints: for each cell type, enough cells must be selected
  for c in cell_types:
    prob += (
      pulp.lpSum(
        cell_type_counts_per_row_group[r].get(c, 0) * x[r]
        for r in range(n_row_groups)
      ) >= needed[c],
      f"cover_{c}"
    )

  # Solve
  prob.solve()

  # Extract selected row groups
  selected = {r for r in range(n_row_groups) if pulp.value(x[r]) > 0.5}
  return selected

In [10]:
def select_row_groups_ilp(cell_type_counts_per_row_group, needed):
  import pulp
  n_row_groups = len(cell_type_counts_per_row_group)
  cell_types = list(needed.keys())
  prob = pulp.LpProblem("RowGroupSelection", pulp.LpMinimize)
  x = [pulp.LpVariable(f"x_{r}", cat="Binary") for r in range(n_row_groups)]
  prob += pulp.lpSum(x)
  for c in cell_types:
    prob += (
      pulp.lpSum(
        cell_type_counts_per_row_group[r].get(c, 0) * x[r]
        for r in range(n_row_groups)
      ) >= needed[c],
      f"cover_{c}"
    )
  prob.solve()
  selected = {r for r in range(n_row_groups) if pulp.value(x[r]) > 0.5}
  return selected

# Test case 1: Simple, non-overlapping
cell_type_counts_per_row_group_1 = [
  {'A': 10},   # row group 0
  {'B': 10},   # row group 1
  {'C': 10},   # row group 2
]
needed_1 = {'A': 5, 'B': 5, 'C': 5}
print("Test 1 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_1, needed_1))
# Expected: {0, 1, 2}

# Test case 2: Overlapping, can use fewer row groups
cell_type_counts_per_row_group_2 = [
  {'A': 5, 'B': 5},   # row group 0
  {'B': 5, 'C': 5},   # row group 1
  {'A': 5, 'C': 5},   # row group 2
]
needed_2 = {'A': 5, 'B': 5, 'C': 5}
print("Test 2 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_2, needed_2))
# Expected: Any two row groups, e.g., {0,1}, {0,2}, or {1,2}

# Test case 3: Need more than one row group for a cell type
cell_type_counts_per_row_group_3 = [
  {'A': 3},   # row group 0
  {'A': 2, 'B': 5},   # row group 1
  {'B': 5, 'C': 5},   # row group 2
]
needed_3 = {'A': 5, 'B': 5}
print("Test 3 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_3, needed_3))
# Expected: {0,1} (for A), and either 1 or 2 for B, so {0,1}, {0,2}, or {0,1,2}

# Test case 4: Impossible to satisfy
cell_type_counts_per_row_group_4 = [
  {'A': 2},   # row group 0
  {'B': 2},   # row group 1
]
needed_4 = {'A': 5, 'B': 5}
print("Test 4 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_4, needed_4))
# Expected: set(), or infeasible (no solution)

# Test case 5: Redundant row group
cell_type_counts_per_row_group_5 = [
  {'A': 5, 'B': 5},   # row group 0
  {'A': 5},           # row group 1
  {'B': 5},           # row group 2
]
needed_5 = {'A': 5, 'B': 5}
print("Test 5 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_5, needed_5))
# Expected: {0}

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/1w/njpw08_93h73169nbj9b9z700000gp/T/35f1856b4d534a05969c7d0b4f752070-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/1w/njpw08_93h73169nbj9b9z700000gp/T/35f1856b4d534a05969c7d0b4f752070-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 21 RHS
At line 25 BOUNDS
At line 29 ENDATA
Problem MODEL has 3 rows, 3 columns and 3 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.5 - 0.00 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from 3 to -1.79769e+308
Probing was tried 0 times and created 0 cuts of

In [11]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import tqdm
import s3fs
from collections import defaultdict
import pyarrow.compute as pc
import numpy as np

fs = s3fs.S3FileSystem(anon=False, profile='xcellerate')


def optimized_row_group_sample_arrow(counts_df: pd.DataFrame, parquet_path: str) -> pd.DataFrame:
  """
  Efficiently sample rows for each cell type by first scanning row groups for relevant cell types and their counts,
  then only reading the minimal set of row groups needed to fulfill the sample requirements.
  Uses Arrow for filtering to minimize memory usage.
  """
  needed = {
    row['cell_type']: row['cell_count']
    for _, row in counts_df.iterrows()
  }
  collected = {cell_type: [] for cell_type in needed}
  done = set()

  print(f"Opening {parquet_path}")
  with (
    fs.open(parquet_path, 'rb')
    if parquet_path[:5] == "s3://"
    else open(parquet_path, 'rb')
  ) as f:
    parquet_file = pq.ParquetFile(f)
    n_row_groups = parquet_file.num_row_groups
    print(f"Found {n_row_groups} row groups")

    if n_row_groups > 1:
      # First pass: build a list of dicts with cell type counts for each row group
      print("Indexing row groups for cell types and counts...")
      cell_type_counts_per_row_group = []
      for rg_idx in tqdm.tqdm(range(n_row_groups)):
        table = parquet_file.read_row_group(
          rg_idx,
          columns=['cell_type', 'assay', 'organism']
        )
        # Arrow filtering
        assay_str = pc.cast(table['assay'], pa.string())
        organism_str = pc.cast(table['organism'], pa.string())
        assay_mask = pc.starts_with(assay_str, '10x')
        organism_mask = pc.equal(organism_str, 'Homo sapiens')
        combined_mask = pc.and_(assay_mask, organism_mask)
        filtered_table = table.filter(combined_mask)
        counts = filtered_table['cell_type'].to_pandas().value_counts().to_dict()
        cell_type_counts_per_row_group.append(counts)
        del table, filtered_table

      # Use ILP to select the optimal set of row groups
      print("Selecting optimal row groups using ILP...")
      row_groups_to_read = select_row_groups_ilp(cell_type_counts_per_row_group, needed)
      del cell_type_counts_per_row_group
      row_groups_to_read = sorted(row_groups_to_read)
    else:
      row_groups_to_read = [ 0 ]

    # Second pass: read only the needed row groups, and sample for each cell type
    print(f"Reading {len(row_groups_to_read)} row groups ...")
    for rg_idx in tqdm.tqdm(row_groups_to_read):
      print("reading row group", rg_idx)
      num_rows = parquet_file.metadata.row_group(rg_idx).num_rows
      print("num rows", num_rows)
      table = parquet_file.read_row_group(
        rg_idx,
        columns=['cell_type', 'assay', 'organism'] + [
          col for col in parquet_file.schema.names
          if col not in ['cell_type', 'assay', 'organism']
        ]
      )

      print("filtering")

      # Arrow filtering
      assay_str = pc.cast(table['assay'], pa.string())
      organism_str = pc.cast(table['organism'], pa.string())
      assay_mask = pc.starts_with(assay_str, '10x')
      organism_mask = pc.equal(organism_str, 'Homo sapiens')
      combined_mask = pc.and_(assay_mask, organism_mask)
      filtered_table = table.filter(combined_mask)
      del table

      # For each cell type still needing samples and present in this row group
      for cell_type in needed:
        if cell_type in done:
          continue
        print("filtering for cell type", cell_type)
        cell_type_mask = pc.equal(filtered_table['cell_type'], cell_type)
        subset_table = filtered_table.filter(cell_type_mask)
        n_needed = needed[cell_type] - sum(len(x) for x in collected[cell_type])
        if n_needed <= 0:
          done.add(cell_type)
          continue
        if subset_table.num_rows > 0:
          n_sample = min(n_needed, subset_table.num_rows)
          # Sample indices in Arrow, then convert only the sample to pandas
          indices = np.random.choice(subset_table.num_rows, n_sample, replace=False)
          sampled_table = subset_table.take(indices)
          sampled_df = sampled_table.to_pandas()
          collected[cell_type].append(sampled_df)
          if n_sample == n_needed:
            done.add(cell_type)
        del subset_table
      # Stop early if all cell types are done
      if len(done) == len(needed):
        print(f"All cell types are done")
        break

  # Concatenate all collected samples
  all_samples = [pd.concat(collected[cell_type], ignore_index=True) for cell_type in collected if collected[cell_type]]
  return pd.concat(all_samples, ignore_index=True)

In [12]:
from pathlib import Path

for index, group in test_counts_pdf.groupby('index'):
  filename = Path(human_only_x10_metadata_pdf.iloc[index].filename)
  print(filename)
  print(group)
  break

s3:/cdiam-h5ad-database/cellxgene_v2/05a49baa-d326-42ae-86d2-94de3a659901
   index         cell_type  cell_count
1     24        fibroblast          74
4     24        macrophage         116
2     24            T cell        1239
7     24            B cell          41
0     24    malignant cell        2000
2     24  endothelial cell         130
1     24          monocyte         121
1     24  mature NK T cell          26
0     24        hepatocyte          23


In [14]:
import s3fs
import tqdm
from dataclasses import dataclass

@dataclass
class EmbeddingFetchError:
  index: int
  file_path: str
  error: Exception
  group: pd.DataFrame

import json
output_dir = data_dir / "cellxgene_embeddings" / "test_v1" 

error_log_path = output_dir / "test_embedding_errors.log"
with open(error_log_path, "a") as error_log:
  fs = s3fs.S3FileSystem(anon=False, profile='xcellerate')
  for index, group in tqdm.tqdm(test_counts_pdf.groupby('index')):
    filename = Path(human_only_x10_metadata_pdf.iloc[index].filename).stem
    file_size = human_only_x10_metadata_pdf.iloc[index].file_size

    print(f"Processing {filename} with size {file_size}")
    
    embedding_path = f"s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/{filename}.parquet"
    
    output_path = output_dir / f"{filename}.parquet"
    if output_path.exists():
      continue

    try:
      test_embeddings_pdf = optimized_row_group_sample_arrow(
        test_counts_pdf[test_counts_pdf["index"] == index],
        embedding_path
      )
      test_embeddings_pdf.to_parquet(output_path)
      del test_embeddings_pdf
    except Exception as e:
      efe = EmbeddingFetchError(index, embedding_path, e, group)
      # Write error info as JSON (convert group to string for simplicity)
      error_log.write(json.dumps({
        "index": efe.index,
        "file_path": efe.file_path,
        "error": str(efe.error),
        "group": efe.group.reset_index(drop=True).to_json()  # or str(efe.group)
      }) + "\n")
      error_log.flush()  # Ensure it's written immediately
      print(efe)
      



  0%|          | 0/15 [00:00<?, ?it/s]

Processing 05a49baa-d326-42ae-86d2-94de3a659901 with size 106964446
Opening s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/05a49baa-d326-42ae-86d2-94de3a659901.parquet
Found 1 row groups
Reading 1 row groups ...


reading row group 0
num rows 4742
filtering
filtering for cell type fibroblast
filtering for cell type macrophage
filtering for cell type T cell
filtering for cell type B cell
filtering for cell type malignant cell
filtering for cell type endothelial cell


  0%|          | 0/1 [00:03<?, ?it/s]

filtering for cell type monocyte
filtering for cell type mature NK T cell
filtering for cell type hepatocyte
All cell types are done


...

filtering for cell type chandelier pvalb GABAergic cortical interneuron
filtering for cell type L5 extratelencephalic projecting glutamatergic cortical neuron
filtering for cell type caudal ganglionic eminence derived interneuron
filtering for cell type vascular leptomeningeal cell
filtering for cell type cerebral cortex endothelial cell
reading row group 1
num rows 50000
filtering


filtering for cell type pvalb GABAergic cortical interneuron
filtering for cell type sst GABAergic cortical interneuron
filtering for cell type chandelier pvalb GABAergic cortical interneuron
reading row group 2
num rows 10752


 67%|██████▋   | 2/3 [01:15<00:37, 37.85s/it]

filtering
filtering for cell type chandelier pvalb GABAergic cortical interneuron
All cell types are done



100%|██████████| 15/15 [05:41<00:00, 22.76s/it]


In [15]:
import src.embeddings as embeddings
test_set_pdf = embeddings.load_data_set(output_dir, "cell_type")
del test_set_pdf